In [1]:
# requires cell_annotation environment
import sys
import os
import io
import json
import importlib
import numpy as np
import collections
import scipy
import sklearn
from pySankey.sankey import sankey # move to plot_utils in future

import seaborn as sns
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.gridspec import GridSpec

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

In [2]:
# reading in util functions:
# notebook directory
current_dir = os.getcwd()

# project directory
root_dir = os.path.abspath(os.path.join(current_dir, '..', '..'))
os.chdir(root_dir)

# for importing utils
sys.path.append(os.path.join(root_dir, 'src', 'functions'))

import annotation_utils
import anno_class
import core_class_test
import classifier_class
import plot_utils

# Reading training and validation data for MAPS

1. Reads in channel information, i.e. which markers are associated with each image slice and creates a dictionary containing each marker(key) and the value representing the range to visualize the image slice.

2. Reads in image (tif file), coordinates of segmentations, and expression information


In [3]:
# read in json
with open('/Users/jabrand2/Desktop/maps/data/resample/resampled_indices.json', 'r') as f:
    loaded_indices = json.load(f)


In [4]:
# framework for annotation:

# data read in:

maps_train = pd.read_csv('/Users/jabrand2/Desktop/maps/data/cell_phenotyping/train.csv')
maps_validation = pd.read_csv('/Users/jabrand2/Desktop/maps/data/cell_phenotyping/valid.csv')

# maps_train.shape # 114,984 training examples
# maps_validation.shape # 28,746 validation samples

In [5]:
maps_train.shape

(114984, 51)

In [6]:
def calculate_metrics(y_true, y_pred, class_names=None):
    """
    Calculates per-class and overall classification metrics and returns them in a pandas DataFrame.

    Args:
        y_true (list or np.array): The ground truth labels.
        y_pred (list or np.array): The predicted labels.
        class_names (list, optional): A list of class names where the index
                                     corresponds to the numerical label. Defaults to None.

    Returns:
        pd.DataFrame: A DataFrame containing per-class and overall metrics.
    """
    # Ensure inputs are numpy arrays for easier indexing and boolean operations
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Get a list of all unique classes from both true and predicted labels
    classes = np.unique(np.concatenate((y_true, y_pred)))
    
    # Initialize dictionaries to store results
    per_class_metrics = {}
    
    # Iterate through each unique class to calculate individual metrics
    for c in classes:
        # Create boolean masks for the current class
        is_class = y_true == c
        is_predicted_class = y_pred == c
        
        # Calculate True Positives (TP), False Positives (FP), False Negatives (FN)
        tp = np.sum(is_class & is_predicted_class)
        fp = np.sum(~is_class & is_predicted_class)
        fn = np.sum(is_class & ~is_predicted_class)
        
        # True Negatives (TN) are instances that are NOT the current class
        # and are also NOT predicted as the current class.
        tn = np.sum(~is_class & ~is_predicted_class)

        # Handle division by zero for precision and recall
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # Calculate F1-score, handling division by zero
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        # Accuracy for a specific class is the number of correct predictions (TP+TN)
        # over the total number of samples. This is a common but sometimes
        # misleading metric for imbalanced classes.
        accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
        
        # Store metrics for the current class
        per_class_metrics[c] = {
            'precision': precision,
            'recall': recall,
            'f1-score': f1_score,
            'accuracy': accuracy,
            'support': np.sum(is_class)
        }
    
    # Create the DataFrame from the per-class metrics dictionary
    metrics_df = pd.DataFrame.from_dict(per_class_metrics, orient='index')

    # Calculate overall metrics (macro-average for precision, recall, and f1-score)
    macro_precision = metrics_df['precision'].mean()
    macro_recall = metrics_df['recall'].mean()
    macro_f1_score = metrics_df['f1-score'].mean()
    
    # Calculate overall accuracy
    overall_correct = np.sum(y_true == y_pred)
    overall_accuracy = overall_correct / len(y_true) if len(y_true) > 0 else 0
    
    # Add an overall row to the DataFrame
    metrics_df.loc['overall'] = [macro_precision, macro_recall, macro_f1_score, overall_accuracy, None]

    # If class names are provided, update the index of the DataFrame
    if class_names is not None:
        # Create a mapping from numerical labels to names
        name_map = {label: name for label, name in enumerate(class_names)}
        
        # Map the current index to the new names, keeping 'overall'
        new_index = [name_map.get(i, i) for i in metrics_df.index]
        metrics_df.index = new_index

    return metrics_df


In [7]:
# the core random forest functionality is used below, it reamins the same as with the full workflow, but without the need for annotation and setting up other class structures

In [8]:
# define class names and color palette for visual results 
class_names = ['B', 'CD4 T', 'CD8 T', 'DC', 'Endothelial', 'Epithelial', 'Lymphatic', 'M1', 'M2', 'Mast', 'Monocyte', 'NK', 'Neutrophil', 'Other', 'Treg', 'Tumor']

# extracting ground truth cell labels for evaluating model performance against ground truth
gt_labels = maps_validation['cell_label'].to_numpy()

In [9]:
i = 0
models = []
model_results = []
for k,v in loaded_indices.items():
    print(k)
    for k2,v2 in v.items():
        
        if k != 'sampled_30':
            continue
            
        i+=1 # change seed through loops
        current_sample_indices = v2
        # Prepare X_train, y_train for the current fold
        X_train_fold = maps_train.iloc[v2, :-1]
        y_train_fold = maps_train.iloc[v2, -1]
    
        # X_test, y_test are fixed (validation set)
        X_test_fixed = maps_validation.iloc[:, :-1]
        y_test_fixed = maps_validation.iloc[:, -1]
    
        # 2. Train and Predict
        # Set random_state for RF for reproducibility per fold/N
        rf = RandomForestClassifier(n_estimators=200, random_state = 5 + i)
        rf.fit(X_train_fold, y_train_fold)

        #if k == 'sampled_30':
        models.append(rf)
        y_pred_fold = rf.predict(X_test_fixed)
    
        y_true_eval = y_test_fixed # Renaming for clarity
    
        # 3. Calculate Metrics
        metrics_report = calculate_metrics(gt_labels, y_pred_fold, class_names = class_names)
        metrics_report['samples_per_class'] = k.split('_')[1]
        metrics_report['fold'] = k.split('_')[1]

        model_results.append(metrics_report) # concatentate when finished
    


sampled_2
sampled_5
sampled_10
sampled_15
sampled_20
sampled_30
sampled_50
sampled_100
sampled_200
sampled_500


In [10]:
rf = models[0]

In [11]:
# Import necessary libraries
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from sklearn.exceptions import UndefinedMetricWarning
import warnings
import matplotlib.cm as cm

unique_labels = np.unique(y_test_fixed)
print(f"Unique class labels found in the validation data: {unique_labels}")

# Get the probability scores for each class from the trained model
# The order of columns in y_score corresponds to the order of classes
# the model was trained on, which is implicitly class_names.
y_score = rf.predict_proba(X_test_fixed)

# Binarize the true labels based on the unique classes present in the validation set
# This prevents the 'No positive class' warning for classes not in the data.
y_test_binarized = label_binarize(y_test_fixed, classes=unique_labels)

# Dictionaries to store precision, recall, and average precision for each class
precision = dict()
recall = dict()
average_precision = dict()

# Suppress the UndefinedMetricWarning for classes not in the validation set
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UndefinedMetricWarning)
    # Calculate precision-recall curve and average precision for each class
    for i in range(len(unique_labels)):
        class_id = unique_labels[i]
        # Calculate precision and recall for the current class
        precision[class_id], recall[class_id], _ = precision_recall_curve(y_test_binarized[:, i], y_score[:, class_id])
        # Calculate average precision for the current class
        average_precision[class_id] = average_precision_score(y_test_binarized[:, i], y_score[:, class_id])

# Plotting the precision-recall curves with a custom style
plt.figure(figsize=(10, 8), facecolor='white') # Set figure background to white
ax = plt.gca()
ax.set_facecolor('white') # Set axes background to white

# Get the 'tab20' colormap and create a list of colors
colors = cm.get_cmap('tab20', len(unique_labels))

# Loop through each class and plot its curve, using the class names and the new color palette
for i, class_id in enumerate(unique_labels):
    class_name = class_names[class_id]
    plt.plot(recall[class_id], precision[class_id], lw=2, color=colors(i),
             label=f'{class_name} (AP = {average_precision[class_id]:0.2f})')

# Add plot titles and labels for clarity
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall")
plt.legend(loc="lower left", fontsize=10)
plt.grid(False)
plt.show()



Unique class labels found in the validation data: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]


/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_99379/3298010929.py:42: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap('tab20', len(unique_labels))


In [15]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from sklearn.exceptions import UndefinedMetricWarning
import warnings
import matplotlib.cm as cm
import matplotlib.pyplot as plt

def rf_pr_curve_wrapper(y_test_fixed, X_test_fixed, rf, class_names, metadata=None):
    """
    Calculates and plots Precision-Recall curves, saving the data to a single
    long-format CSV file with a unique name based on metadata.

    Args:
        y_test_fixed (array-like): Ground truth labels.
        X_test_fixed (array-like): Test features.
        rf (object): A trained RandomForestClassifier model.
        class_names (list): A list of class names.
        metadata (dict, optional): A dictionary of metadata to include in the filename.
    """
    # Calculate PR values (your original code)
    unique_labels = np.unique(y_test_fixed)
    y_score = rf.predict_proba(X_test_fixed)
    y_test_binarized = label_binarize(y_test_fixed, classes=unique_labels)
    precision, recall, average_precision = dict(), dict(), dict()
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UndefinedMetricWarning)
        for i in range(len(unique_labels)):
            class_id = unique_labels[i]
            precision[class_id], recall[class_id], _ = precision_recall_curve(
                y_test_binarized[:, i], y_score[:, class_id]
            )
            average_precision[class_id] = average_precision_score(
                y_test_binarized[:, i], y_score[:, class_id]
            )

    # Plotting logic (from your original code)
    plt.figure(figsize=(10, 8), facecolor='white')
    ax = plt.gca()
    ax.set_facecolor('white')
    colors = cm.get_cmap('tab20', len(unique_labels))
    for i, class_id in enumerate(unique_labels):
        class_name = class_names[class_id]
        plt.plot(recall[class_id], precision[class_id], lw=2, color=colors(i),
                 label=f'{class_name} (AP = {average_precision[class_id]:0.2f})')
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall")
    plt.legend(loc="lower left", fontsize=10)
    plt.grid(False)
    plt.show()

    # Create and save a long-format DataFrame with metadata
    long_format_data = []
    for i, class_id in enumerate(unique_labels):
        class_name = class_names[class_id]
        for r, p in zip(recall[class_id], precision[class_id]):
            data_point = {
                'model_type': 'RandomForest',  # Tag the model type for comparison
                'curve_type': 'per_class',
                'class_name': class_name,
                'precision': p,
                'recall': r
            }
            if metadata:
                data_point.update(metadata)
            long_format_data.append(data_point)
    
    df = pd.DataFrame(long_format_data)
    
    filename_prefix = ""
    if metadata:
        metadata_str = "_".join(f"{k}_{v}" for k, v in metadata.items())
        filename_prefix = f"{metadata_str}_"

    output_filename = f'{filename_prefix}rf_pr_curve_data.csv'
    df.to_csv(output_filename, index=False)
    print(f"All precision-recall data has been saved to '{output_filename}'.")

In [13]:
# Assuming your loop and data variables are defined

i = 0
models = []
model_results = []
for k,v in loaded_indices.items():
    print(k)
    for k2,v2 in v.items():
        
        if k != 'sampled_30':
            continue
            
        i+=1 # change seed through loops
        current_sample_indices = v2
        # Prepare X_train, y_train for the current fold
        X_train_fold = maps_train.iloc[v2, :-1]
        y_train_fold = maps_train.iloc[v2, -1]
    
        # X_test, y_test are fixed (validation set)
        X_test_fixed = maps_validation.iloc[:, :-1]
        y_test_fixed = maps_validation.iloc[:, -1]
    
        # 2. Train and Predict
        # Set random_state for RF for reproducibility per fold/N
        rf = RandomForestClassifier(n_estimators=200, random_state = 5 + i)
        rf.fit(X_train_fold, y_train_fold)

        #if k == 'sampled_30':
        models.append(rf)
        y_pred_fold = rf.predict(X_test_fixed)
    
        y_true_eval = y_test_fixed # Renaming for clarity
  
        # Extract metadata from the model name
        samples_per_class = k.split('_')[1]
        fold = k2.split('_')[1]
        
        # Create the metadata dictionary
        metadata = {
            'samples_per_class': samples_per_class,
            'fold': fold
        }
        
        # Load your trained Random Forest model and data
        # rf = ... (your trained Random Forest model)
        # X_test_fixed, y_test_fixed = ... (your test data)
        # class_names = ... (your list of class names)

        # Call the new wrapper with the metadata
        rf_pr_curve_wrapper(
            y_test_fixed=y_test_fixed,
            X_test_fixed=X_test_fixed,
            rf=rf,
            class_names=class_names,
            metadata=metadata
        )

sampled_2
sampled_5
sampled_10
sampled_15
sampled_20
sampled_30


/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_99379/121827750.py:44: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap('tab20', len(unique_labels))


All precision-recall data has been saved to 'samples_per_class_30_fold_1_rf_pr_curve_data.csv'.


/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_99379/121827750.py:44: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap('tab20', len(unique_labels))


All precision-recall data has been saved to 'samples_per_class_30_fold_2_rf_pr_curve_data.csv'.


...

All precision-recall data has been saved to 'samples_per_class_30_fold_3_rf_pr_curve_data.csv'.


/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_99379/121827750.py:44: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap('tab20', len(unique_labels))


All precision-recall data has been saved to 'samples_per_class_30_fold_4_rf_pr_curve_data.csv'.


/var/folders/x5/3q825t5d60l4ksl3q5gknfjxmmfx08/T/ipykernel_99379/121827750.py:44: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = cm.get_cmap('tab20', len(unique_labels))


All precision-recall data has been saved to 'samples_per_class_30_fold_5_rf_pr_curve_data.csv'.
sampled_50
sampled_100
sampled_200
sampled_500


In [14]:
num_classes = len(unique_labels)
num_models = 5
ncols = 4
nrows = (num_classes + ncols - 1) // ncols

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 5 * nrows))
axes = axes.flatten()

# Get a color palette for the different models
model_colors = plt.get_cmap('Paired', num_models)

# Loop through each unique class to create a dedicated subplot
for class_index, class_id in enumerate(unique_labels):
    ax = axes[class_index]
    class_name = class_names[class_id]
    
    # Loop through each model and plot its curve
    for fold_num, model in enumerate(models):
        # Get the probability scores for this model
        y_score = model.predict_proba(X_test_fixed)
        
        # Binarize the true labels for the current class
        y_test_binarized = label_binarize(y_test_fixed, classes=unique_labels)
        
        # Suppress UndefinedMetricWarning if a class has no positive samples
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UndefinedMetricWarning)
            
            # Calculate precision, recall, and average precision
            precision, recall, _ = precision_recall_curve(y_test_binarized[:, class_index], y_score[:, class_index])
            average_precision = average_precision_score(y_test_binarized[:, class_index], y_score[:, class_index])
        
        # Plot the curve for this model and class
        ax.plot(recall, precision, lw=1.5, color=model_colors(fold_num),
                label=f'Fold {fold_num + 1} (AP = {average_precision:0.2f})')
    
    # Set subplot titles and labels
    ax.set_title(f'Precision-Recall for {class_name}')
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left", fontsize = 6)
    ax.grid(False)

# Hide any unused subplots
for j in range(num_classes, len(axes)):
    fig.delaxes(axes[j])

# Adjust subplot layout
plt.subplots_adjust(
    left=0.1,
    right=0.9,
    top=0.9,
    bottom=0.1,
    wspace=0.6,
    hspace=0.8
)

# Show the plot
plt.show()

In [19]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from sklearn.exceptions import UndefinedMetricWarning
import warnings
import matplotlib.cm as cm

class_names = ['B', 'CD4 T', 'CD8 T', 'DC', 'Endothelial', 'Epithelial', 'Lymphatic', 'M1', 'M2', 'Mast', 'Monocyte', 'NK', 'Neutrophil', 'Other', 'Treg', 'Tumor']
unique_labels = np.arange(len(class_names))



plt.figure(figsize=(10, 8))
plt.title("Micro-Averaged Precision-Recall Curve (All Models)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.grid(False)

# Get a color palette for the different models
model_colors = plt.get_cmap('Paired', 5)

# Binarize the true labels once, since the test data is fixed
y_test_binarized = label_binarize(y_test_fixed, classes=unique_labels)

# Loop through each model and plot its global curve
for fold_num, model in enumerate(models):
    # Get the probability scores for this model
    y_score = model.predict_proba(X_test_fixed)
    
    # Calculate micro-averaged precision, recall, and average precision
    # The 'ravel()' function flattens the arrays to a 1D format for global averaging
    precision_micro, recall_micro, _ = precision_recall_curve(y_test_binarized.ravel(), y_score.ravel())
    average_precision_micro = average_precision_score(y_test_binarized, y_score, average="micro")
    
    # Plot the micro-averaged curve for this model
    plt.plot(recall_micro, precision_micro, lw=1.5, color=model_colors(fold_num),
             label=f'Fold {fold_num + 1} (AP = {average_precision_micro:0.2f})')

# Add legend and show the plot
plt.legend(loc="lower left", fontsize=10)
plt.show()
